* Deep networks can fail because numbers become too small, too large, or poorly scaled.

* Initialization chooses starting parameter values to make training more stable.

In [1]:
import math
import random
import numpy as np
import torch

# 5.4.1 Vanishing and Exploding Gradients

## 1. Intuition

* A vanishing gradient becomes extremely small as it moves backward through layers. An exploding gradient becomes extremely large.
> * A large gradient means that changing the param would have a relatively large effect on loss
> * A small gradient can mean that the model fits the example well / small prediction errors / not sensitive to it

* Both problems make learning difficult because parameter updates become too tiny or too unstable.

## 2. Why this exists

* Deep networks multiply many local derivatives. Repeated multiplication can shrink or grow gradients rapidly.

## 3. Examples

* Repeated multiplication can shrink values.

In [4]:
value = torch.tensor(1.0)

for _ in range(5):
  value = value * 0.2 # iterated over 5 times -> 1*0.2^(5) = 0.00032

value

tensor(0.0003)

* Repeated multiplication can grow values.

In [5]:
value = torch.tensor(1.0)

for _ in range(5):
  value = value * 3 # iterated over 5 times -> 1*3^(5) = 243

value

tensor(243.)

## 4. Step-by-step breakdown

* The 1st loop repeatedly multiplies by a number < 1, so the value shrinks.

* The 1st loop repeatedly multiplies by a number > 1, so the value grows.

* Backpropagation can face similar repeated multiplication across layers.

## 5. Connection to ML systems

* Stability problems motivate careful activation choices, initialization schemes, normalization, and gradient clipping.

    ## Training Stability

    | Technique | Common Choices | Main Purpose |
    |---|---|---|
    | **Activation** | ReLU, Leaky ReLU, Sigmoid, Tanh, GELU, SiLU | Controls how values and gradients propagate |
    | **Initialization** | Xavier/Glorot, He/Kaiming | Sets a healthy starting scale for weights |
    | **Normalization** | BatchNorm, LayerNorm, RMSNorm | Keeps intermediate activations at useful scales |
    | **Gradient Clipping** | `clip_grad_norm_`, `clip_grad_value_` | Limits excessively large gradients |

## 6. Common confusion points

- Vanishing gradients slow or stop early-layer learning.
- Exploding gradients can produce unstable updates.
- Depth increases the chance of repeated-scaling problems as more layers repeatedly transform activations and gradients.
- Stable training is an engineering and mathematical concern.

# 5.4.2 Parameter Initialization

## 1. Intuition

* Parameter initialization means choosing starting values for weights and biases before training.

* Random initialization breaks symmetry. Symmetry means 2 units start and behave identically, learning the same thing.

## 2. Why this exists

* Bad initialization can make activations or gradients too large or too small before training even has a chance.

## 3. Examples

* Compare zero and random weight initialization.

In [8]:
zeros = torch.zeros(2, 3)
random_weights = torch.randn(2, 3) * 0.01
zeros, random_weights

(tensor([[0., 0., 0.],
         [0., 0., 0.]]),
 tensor([[ 0.0055, -0.0096, -0.0050],
         [ 0.0076, -0.0065, -0.0121]]))

* Use Xavier initialization on a linear layer.

In [9]:
layer = torch.nn.Linear(4, 3)

torch.nn.init.xavier_uniform_(layer.weight) # Randomly initializes weights uniformly within a range determined by the layer's input/output sizes
                                            # The more inputs a neuron combines, the more carefully you need to control the typical size of each weight (e.g. bigger layer/each weight will be smaller and vice versa)
torch.nn.init.zeros_(layer.bias)

layer.weight.shape, layer.bias.shape

(torch.Size([3, 4]), torch.Size([3]))

## 4. Step-by-step breakdown

* Zero weights can make hidden units identical.

* Small random weights break that identical behavior.

* Xavier initialization chooses a scale based on input and output sizes.

* The underscore in `xavier_uniform_` means the function modifies the tensor in place.

## 5. Connection to ML systems

* PyTorch layers initialize parameters automatically, but explicit initialization is common in research and debugging.

## 6. Common confusion points

- Initialization happens before training.
- Random does not mean arbitrary; scale matters.
- Biases are often initialized to zero.
- In-place initialization functions commonly end with `_` in PyTorch.

# 5.4.3 Summary

## 1. Intuition

* Numerical stability is about keeping values and gradients in useful ranges.

* Initialization is 1 early decision that affects stability.

## 2. Why this exists

* A network that starts in a bad numerical regime may train slowly or fail entirely.

## 3. Examples

* A simple stability checklist.

  - **Activation scale** → Are intermediate values becoming extremely large or small?
  - **Gradient scale** → Are gradients becoming extremely large or small?
  - **Weight initialization** → Did the parameters start at a reasonable scale?
  - **Learning rate** → Are parameter updates too large or too small?

## 4. Step-by-step breakdown

* The checklist names common sources of instability.

* Activations are forward-pass values.

* Gradients are backward-pass values.

* Both interact with initialization and learning rate.

## 5. Connection to ML systems

* Deep learning practice often starts by checking whether loss is finite and gradients are reasonable.

## 6. Common confusion points

- Stable numbers are required for useful learning.
- Initialization is not a replacement for good data or architecture.
- Larger networks need more careful stability checks.
- Watch for `nan` or `inf` values during debugging.

# 5.4.4 Exercises

## 1. Intuition

* These exercises practice recognizing scale problems and applying initialization.

## 2. Why this exists

* Small experiments reveal why initialization matters before large models obscure the issue.

## 3. Examples

* Exercise 1: observe repeated shrinking.

In [12]:
x = torch.tensor(1.0)

for _ in range(4):
  x = x * 0.5 # 1*0.5^(4) = 0.0625

x

tensor(0.0625)

* Exercise 2: initialize a layer bias to zero.

In [14]:
layer = torch.nn.Linear(3, 2)
torch.nn.init.zeros_(layer.bias)
layer.bias # shape of (2, ) since bias is 1 value per output feature

Parameter containing:
tensor([0., 0.], requires_grad=True)

## 4. Step-by-step breakdown

* Exercise 1 shows repeated scaling below 1.

* Exercise 2 uses an in-place initializer.

* Both are tiny versions of stability-related operations.

## 5. Connection to ML systems

* These ideas become more important as networks deepen.

## 6. Common confusion points

- Repeated multiplication can change scale quickly.
- Initialization functions can modify parameters directly.
- Bias and weight initialization can use different rules (e.g. weights with Xavier and bias with zeros).
- Stability should be checked early in training.